# Pilot Runs — Colab Direct, No Google Drive

## What we are doing

Running the small-model pilot training from an ephemeral Colab runtime.

## Where to run

**Google Colab T4 GPU**

No Google Drive mount is used.

## You need to upload

1. `pilot_runs_scaffold_colab_direct.zip`
2. tokenizer artifact: `toolcall_spm_32k.model` or a zip containing it
3. 50M smoke shard zip containing `manifest.json`, `manifest_shards.jsonl`, and `shards/shard_*.bin`


## 1. Check GPU

In [1]:
!nvidia-smi

Mon Jul 20 17:11:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Extract the pilot scaffold

In [3]:
from pathlib import Path
import zipfile
import shutil

# Colab base directory
BASE_DIR = Path("/content")

# Final pilot_runs directory
PILOT_DIR = BASE_DIR / "pilot_runs"

print("Base directory:", BASE_DIR)
print("Target pilot directory:", PILOT_DIR)

# The zip is already uploaded to /content.
exact_zip = BASE_DIR / "pilot_runs_scaffold_colab_direct.zip"

if exact_zip.exists():
    scaffold_zip = exact_zip
else:
    # Fallback: find any matching scaffold zip in /content.
    zip_candidates = sorted(BASE_DIR.glob("*pilot*runs*scaffold*.zip"))

    if not zip_candidates:
        zip_candidates = sorted(BASE_DIR.glob("*.zip"))

    if not zip_candidates:
        raise FileNotFoundError(
            "Could not find pilot_runs_scaffold_colab_direct.zip in /content. "
            "Check the Colab file browser and confirm the zip is uploaded."
        )

    scaffold_zip = zip_candidates[0]

print("Using scaffold zip:", scaffold_zip)

# Temporary extraction directory
extract_root = BASE_DIR / "pilot_scaffold_extract"

if extract_root.exists():
    shutil.rmtree(extract_root)

extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(scaffold_zip, "r") as zf:
    zf.extractall(extract_root)

# Detect whether the zip has a top-level folder or files directly.
required = ["requirements.txt", "scripts", "configs"]

possible_roots = [extract_root] + [
    p for p in extract_root.iterdir()
    if p.is_dir()
]

source_root = None

for candidate in possible_roots:
    if all((candidate / item).exists() for item in required):
        source_root = candidate
        break

if source_root is None:
    print("Extracted files:")
    for p in sorted(extract_root.rglob("*"))[:100]:
        print(" -", p)

    raise RuntimeError(
        "Could not find scaffold root with requirements.txt, scripts/, and configs/."
    )

print("Detected scaffold root:", source_root)

# Recreate /content/pilot_runs cleanly.
if PILOT_DIR.exists():
    shutil.rmtree(PILOT_DIR)

shutil.copytree(source_root, PILOT_DIR)

print("\nPilot scaffold ready:", PILOT_DIR)
print("Contents:")

for p in sorted(PILOT_DIR.iterdir()):
    print(" -", p.name)

# Optional cleanup of temporary extraction folder
shutil.rmtree(extract_root)

print("\nDone.")

Base directory: /content
Target pilot directory: /content/pilot_runs
Using scaffold zip: /content/pilot_runs_scaffold_colab_direct.zip
Detected scaffold root: /content/pilot_scaffold_extract

Pilot scaffold ready: /content/pilot_runs
Contents:
 - .gitignore
 - README.md
 - configs
 - notebooks
 - requirements.txt
 - results
 - runs
 - scripts

Done.


## 3. Install requirements

In [5]:
%cd /content/pilot_runs
!pip -q install -r requirements.txt


/content/pilot_runs



## 4. Tokenizer artifact

In [7]:
from pathlib import Path
import shutil

# Colab base directory
BASE_DIR = Path("/content")

# This matches the config path when running from /content/pilot_runs:
# ../tokenizer/artifacts/tokenizer/toolcall_spm_32k.model
TOKENIZER_ARTIFACT_DIR = BASE_DIR / "tokenizer" / "artifacts" / "tokenizer"
TOKENIZER_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Tokenizer destination:", TOKENIZER_ARTIFACT_DIR)

# Expected already-uploaded files
source_model = BASE_DIR / "toolcall_spm_32k.model"
source_vocab = BASE_DIR / "toolcall_spm_32k.vocab"
source_manifest = BASE_DIR / "tokenizer_manifest.json"

# Fallback search under /content
if not source_model.exists():
    model_matches = sorted(BASE_DIR.rglob("toolcall_spm_32k.model"))
    if model_matches:
        source_model = model_matches[0]

if not source_model.exists():
    raise FileNotFoundError(
        "Could not find toolcall_spm_32k.model under /content. "
        "Check the Colab file browser and confirm the .model file is uploaded."
    )

print("Found model:", source_model)

# Copy required tokenizer model
dest_model = TOKENIZER_ARTIFACT_DIR / "toolcall_spm_32k.model"
shutil.copy2(source_model, dest_model)

# Copy optional vocab if present
if source_vocab.exists():
    shutil.copy2(source_vocab, TOKENIZER_ARTIFACT_DIR / "toolcall_spm_32k.vocab")
    print("Found vocab:", source_vocab)
else:
    vocab_matches = sorted(BASE_DIR.rglob("toolcall_spm_32k.vocab"))
    if vocab_matches:
        shutil.copy2(vocab_matches[0], TOKENIZER_ARTIFACT_DIR / "toolcall_spm_32k.vocab")
        print("Found vocab:", vocab_matches[0])
    else:
        print("Warning: toolcall_spm_32k.vocab not found. The .model file is enough.")

# Copy optional manifest if present
if source_manifest.exists():
    shutil.copy2(source_manifest, TOKENIZER_ARTIFACT_DIR / "tokenizer_manifest.json")
    print("Found manifest:", source_manifest)
else:
    manifest_matches = sorted(BASE_DIR.rglob("tokenizer_manifest.json"))
    if manifest_matches:
        shutil.copy2(manifest_matches[0], TOKENIZER_ARTIFACT_DIR / "tokenizer_manifest.json")
        print("Found manifest:", manifest_matches[0])
    else:
        print("Warning: tokenizer_manifest.json not found. This is okay for the pilot run.")

MODEL_PATH = dest_model

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer model was not copied correctly: {MODEL_PATH}"
    )

print("\nTokenizer ready:", MODEL_PATH)
print("Files:")

for p in sorted(TOKENIZER_ARTIFACT_DIR.iterdir()):
    size_mb = p.stat().st_size / 1024**2
    print(" -", p.name, round(size_mb, 3), "MB")

Tokenizer destination: /content/tokenizer/artifacts/tokenizer
Found model: /content/toolcall_spm_32k.model

Tokenizer ready: /content/tokenizer/artifacts/tokenizer/toolcall_spm_32k.model
Files:
 - toolcall_spm_32k.model 0.728 MB


## 5. Extract the 50M smoke shard zip



In [8]:
from pathlib import Path
import shutil

# Colab base directory
BASE_DIR = Path("/content")

# This matches the config path when running from /content/pilot_runs:
# ../tokenizer/data/tokenized/smoke_50m
SMOKE_DIR = BASE_DIR / "tokenizer" / "data" / "tokenized" / "smoke_50m"
SHARDS_OUT = SMOKE_DIR / "shards"

# Existing source locations
SOURCE_MANIFEST = BASE_DIR / "manifest.json"
SOURCE_MANIFEST_SHARDS = BASE_DIR / "manifest_shards.jsonl"
SOURCE_SHARDS_DIR = BASE_DIR / "shards"

print("Expected source manifest:", SOURCE_MANIFEST)
print("Expected source shard manifest:", SOURCE_MANIFEST_SHARDS)
print("Expected source shards dir:", SOURCE_SHARDS_DIR)

# Validate source files
if not SOURCE_MANIFEST.exists():
    raise FileNotFoundError(
        "manifest.json not found at /content/manifest.json"
    )

if not SOURCE_MANIFEST_SHARDS.exists():
    raise FileNotFoundError(
        "manifest_shards.jsonl not found at /content/manifest_shards.jsonl"
    )

if not SOURCE_SHARDS_DIR.exists():
    raise FileNotFoundError(
        "shards directory not found at /content/shards"
    )

source_shards = sorted(SOURCE_SHARDS_DIR.glob("shard_*.bin"))

if not source_shards:
    raise FileNotFoundError(
        "No shard_*.bin files found under /content/shards"
    )

print("\nFound source shards:")
for shard in source_shards:
    print(" -", shard.name, round(shard.stat().st_size / 1024**2, 2), "MB")

# Recreate destination smoke_50m directory cleanly
if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR)

SHARDS_OUT.mkdir(parents=True, exist_ok=True)

# Copy manifests
shutil.copy2(SOURCE_MANIFEST, SMOKE_DIR / "manifest.json")
shutil.copy2(SOURCE_MANIFEST_SHARDS, SMOKE_DIR / "manifest_shards.jsonl")

# Copy shard binaries
for shard in source_shards:
    shutil.copy2(shard, SHARDS_OUT / shard.name)

print("\nSmoke shard ready:", SMOKE_DIR)
print("Shard count:", len(list(SHARDS_OUT.glob("shard_*.bin"))))

total_size_mb = sum(
    p.stat().st_size
    for p in SHARDS_OUT.glob("shard_*.bin")
) / 1024**2

print("Total shard size MB:", round(total_size_mb, 2))

print("\nDestination contents:")
for p in sorted(SMOKE_DIR.iterdir()):
    print(" -", p.name)

print("\nDestination shards:")
for p in sorted(SHARDS_OUT.glob("shard_*.bin")):
    print(" -", p.name, round(p.stat().st_size / 1024**2, 2), "MB")

Expected source manifest: /content/manifest.json
Expected source shard manifest: /content/manifest_shards.jsonl
Expected source shards dir: /content/shards

Found source shards:
 - shard_00000.bin 19.07 MB
 - shard_00001.bin 19.07 MB
 - shard_00002.bin 19.07 MB
 - shard_00003.bin 19.07 MB
 - shard_00004.bin 19.07 MB

Smoke shard ready: /content/tokenizer/data/tokenized/smoke_50m
Shard count: 5
Total shard size MB: 95.37

Destination contents:
 - manifest.json
 - manifest_shards.jsonl
 - shards

Destination shards:
 - shard_00000.bin 19.07 MB
 - shard_00001.bin 19.07 MB
 - shard_00002.bin 19.07 MB
 - shard_00003.bin 19.07 MB
 - shard_00004.bin 19.07 MB


## 6. Inspect pilot inputs

In [9]:
%cd /content/pilot_runs

!python scripts/inspect_pilot_inputs.py \
  --data-dir /content/tokenizer/data/tokenized/smoke_50m \
  --tokenizer /content/tokenizer/artifacts/tokenizer/toolcall_spm_32k.model \
  --sample-tokens 256

/content/pilot_runs
Tokenizer: /content/tokenizer/artifacts/tokenizer/toolcall_spm_32k.model
Vocab size: 32000
Data dir: /content/tokenizer/data/tokenized/smoke_50m
Shards: 5
shard_00000.bin: 10,000,000 tokens, max_id=31999
shard_00001.bin: 10,000,000 tokens, max_id=31999
shard_00002.bin: 10,000,000 tokens, max_id=31999
shard_00003.bin: 10,000,000 tokens, max_id=31997
shard_00004.bin: 10,000,000 tokens, max_id=31994

Total tokens: 50,000,000
Global max token id: 31999

Decoded sample:
The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and 

## 7. Quick 100-step pilot

Run this first. It is only to catch runtime errors quickly.

In [10]:
%cd /content/pilot_runs

!python scripts/train_pilot_lm.py \
  --config configs/tiny_smoke_50m_quick_100_steps.json


/content/pilot_runs
/content/pilot_runs/scripts/train_pilot_lm.py:258: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
{
  "run_config_path": "configs/tiny_smoke_50m_quick_100_steps.json",
  "data_dir": "../tokenizer/data/tokenized/smoke_50m",
  "tokenizer": "../tokenizer/artifacts/tokenizer/toolcall_spm_32k.model",
  "out_dir": "runs/tiny_smoke_50m_quick_100_steps",
  "model_config": {
    "vocab_size": 32000,
    "seq_len": 512,
    "hidden_size": 256,
    "num_layers": 6,
    "num_heads": 4,
    "num_kv_heads": 2,
    "intermediate_size": 768,
    "dropout": 0.0,
    "rope_base": 10000.0
  },
  "parameters": 12913920,
  "dataset_tokens": 50000000,
  "seq_len": 512,
  "batch_size": 8,
  "grad_accum": 4,
  "effective_batch_tokens": 16384,
  "max_steps": 100,
  "learning_rate": 0.0003,
  "device": "cuda",
  "seed": 42
}
/content/pilot_runs

## 8. Full 1000-step tiny smoke pilot

Run this only after the 100-step pilot succeeds.

In [11]:
%cd /content/pilot_runs

!python scripts/train_pilot_lm.py \
  --config configs/tiny_smoke_50m.json


/content/pilot_runs
/content/pilot_runs/scripts/train_pilot_lm.py:258: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
{
  "run_config_path": "configs/tiny_smoke_50m.json",
  "data_dir": "../tokenizer/data/tokenized/smoke_50m",
  "tokenizer": "../tokenizer/artifacts/tokenizer/toolcall_spm_32k.model",
  "out_dir": "runs/tiny_smoke_50m",
  "model_config": {
    "vocab_size": 32000,
    "seq_len": 512,
    "hidden_size": 256,
    "num_layers": 6,
    "num_heads": 4,
    "num_kv_heads": 2,
    "intermediate_size": 768,
    "dropout": 0.0,
    "rope_base": 10000.0
  },
  "parameters": 12913920,
  "dataset_tokens": 50000000,
  "seq_len": 512,
  "batch_size": 8,
  "grad_accum": 4,
  "effective_batch_tokens": 16384,
  "max_steps": 1000,
  "learning_rate": 0.0003,
  "device": "cuda",
  "seed": 42
}
/content/pilot_runs/scripts/train_pilot_lm.py:291:

## 9. Summarize and download result

In [14]:
from google.colab import files
from pathlib import Path
import shutil

%cd /content/pilot_runs

!python scripts/summarize_pilot_run.py \
  --run-dir runs/tiny_smoke_50m \
  --out results/tiny_smoke_50m_result.md

print(Path("results/tiny_smoke_50m_result.md").read_text())

archive_path = shutil.make_archive(
    "/content/pilot_tiny_smoke_results",
    "zip",
    root_dir="/content/pilot_runs",
    base_dir="results",
)

print("Downloading:", archive_path)
files.download(archive_path)


/content/pilot_runs
Wrote: results/tiny_smoke_50m_result.md
# Pilot Run Summary

Run directory: `runs/tiny_smoke_50m`

## Setup

- Data dir: `../tokenizer/data/tokenized/smoke_50m`
- Tokenizer: `../tokenizer/artifacts/tokenizer/toolcall_spm_32k.model`
- Device: `cuda`
- Parameters: `12,913,920`
- Dataset tokens: `50,000,000`
- Sequence length: `512`
- Batch size: `8`
- Gradient accumulation: `4`
- Effective batch tokens: `16,384`
- Max steps: `1000`
- Learning rate: `0.0003`

## Result

- First logged step: `1`
- First loss: `41.7056`
- Last logged step: `1000`
- Last loss: `12.6615`
- Loss change: `-29.0441`
- Last tokens/sec: `65,069`

## Interpretation

- Training loop completed: `TODO`
- Loss decreased meaningfully: `TODO`
- CUDA OOM: `TODO`
- Checkpoints saved: `TODO`
- Generation sample usable for sanity check: `TODO`

## Notes

TODO: Add observations from Colab logs.

Downloading: /content/pilot_tiny_smoke_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>